In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
import matplotlib.pyplot as plt
import numpy as np
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'


# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

for i in range(10,13):
    img, label = train_dataset[i] # Changed mask to label
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img.permute(1, 2, 0))  # Convert (C, H, W) to (H, W, C)
    axes[0].set_title("Image")
    axes[0].axis("off")
    # Display the character label instead of trying to show a non-existent mask
    axes[1].text(0.5, 0.5, letters[label - 1], fontsize=50, ha='center', va='center')
    axes[1].set_title("Label")
    axes[1].axis("off")
    plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
import torch
import torchvision.models as models
from torch import nn
# from tqdm import tqdm # tqdm not needed for model setup itself if not iterating

# Load pretrained EfficientNetV2-S model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
model.to(device)

# Freeze the backbone (feature extractor)
for param in model.features.parameters():
    param.requires_grad = False

num_ftrs = model.classifier[1].in_features

# Replace the classifier with a new one for 26 classes

model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(num_ftrs, num_classes)
)

model = model.to(device) # Ensure the modified model is on the correct device

print(f"Model architecture:\n{model}")
print(f"\nNumber of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")




In [ ]:
# Write your code here
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)
        labels = labels - 1  # Adjust labels from 1-26 to 0-25 for CrossEntropyLoss

        outputs = model(images)  # Output shape: [batch_size, num_classes]
        loss = criterion(outputs, labels)

        optimizer.zero_grad()  #clear old gradients
        loss.backward()  #compute new gradients
        optimizer.step()  #update weights

        total_loss += loss.item()

        # Track accuracy for multi-class classification
        _, predictions = torch.max(outputs, 1) # Get the index of the max log-probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            labels = labels - 1  # Adjust EMNIST labels from 1-26 to 0-25 for CrossEntropyLoss

            outputs = model(images) # Output shape: [batch_size, num_classes]
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Compute accuracy for multi-class classification
            _, predictions = torch.max(outputs, 1) # Get the index of the max log-probability
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [ ]:
# Write your code here
import torch.optim as optim

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 2


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device) # Use test_loader for validation

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

In [ ]:
# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
# Write your code here
